In [1]:
import random
import numpy as np
from tqdm import tqdm

In [361]:
class Maze:
    def __init__(self):
        self.maze = [
            "OOOOOOOCG",
            "OOCOOOOCO",
            "SOCOOOOCO",
            "OOCOOOOOO",
            "OOOOOCOOO",
            "OOOOOOOOO",
        ]
        self.starting_state = (2,0)
        self.goal_state = (0, 8)

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

        self.terminal_states = [(1,2), (2, 2), (3, 2), (0, 7), (1, 7), (2, 7), (4, 5), (0,8)]

    def get_action_space(self):
        return ["up", "down", "left", "right"]

    def print_maze(self):
        print(self.maze)

    def change_state(self, action):

        new_state = (0, 0)

        if action == "up":
            if self.current_state[0] == 0:
                return
            new_state = (self.current_state[0] - 1, self.current_state[1])
        elif action == "down":
            if self.current_state[0] == 5:
                return
            new_state = (self.current_state[0] + 1, self.current_state[1])
        elif action == "left":
            if self.current_state[1] == 0:
                return
            new_state = (self.current_state[0], self.current_state[1] - 1)
        elif action == "right":
            if self.current_state[1] == 8:
                return

            new_state = (self.current_state[0], self.current_state[1] + 1)

        if (
            self.maze[new_state[0]][new_state[1]] == "O"
            or self.maze[new_state[0]][new_state[1]] == "S"
        ):
            self.current_state = new_state
        elif self.maze[new_state[0]][new_state[1]] == "G":
            self.current_state = new_state
            # print("Goal reached!")

    def step(self, action):

        if self.terminated or self.truncated:
            observation = self.current_state
            reward = 0
            return observation, reward, self.terminated, self.truncated

        self.number_of_steps += 1

        self.change_state(action)

        observation = self.current_state

        reward = 0
        if self.maze[self.current_state[0]][self.current_state[1]] == "G":
            reward = 1

            self.terminated  = 1
        else:
            reward = 0

        if self.number_of_steps == 100000:
            self.truncated = 1

        return observation, reward, self.terminated, self.truncated

    def reset(self):
        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

In [362]:
class Dyna_Q:
    def __init__(self, env, n, number_of_episodes = 100):
        self.states_and_actions_taken = {}
        self.env = env
        self.n = n

        self.Q = {}
        self.Model = {}

        self.initialize_Q_and_Model()

        self.number_of_episodes = number_of_episodes

        self.trajectories = []

    def initialize_Q_and_Model(self):

        for row in range(6):
            for col in range(9):
                self.Q[(row, col)] = {"up": 0, "down": 0, "left": 0, "right": 0}
                self.Model[(row, col)] = {"up": (0,0), "down": (0,0), "left": (0,0), "right": (0,0)}


    def get_argmax(self, S):
        state_dict = self.Q[S]

        max_value = max(state_dict.values())
        return [action for action, value in state_dict.items() if value == max_value]



    def get_max(self, S):
        state_dict = self.Q[S]

        action = max(state_dict, key=state_dict.get)
        return self.Q[S][action]


    def get_action(self, epsilon, S):
        action = random.choice(self.env.get_action_space())
        if random.random() > epsilon:
            possible_actions = self.get_argmax(S)
            action = random.choice(possible_actions)

        return action

    def update_state_action_collection(self, S, A):
        if S not in self.states_and_actions_taken:
            self.states_and_actions_taken[S] = []

        if A not in self.states_and_actions_taken[S]:
            self.states_and_actions_taken[S].append(A)

    def get_policy(self):
        policy = {}
        terminal_states = self.env.terminal_states
        for S in self.Q:

            if S in terminal_states:
                policy[S] = "Terminal"
                continue

            A = self.get_argmax(S)[0]




            policy[S] = A
        return policy




    def do_planning_updates(self):
        for _ in range(self.n):

            S = random.choice(list(self.states_and_actions_taken.keys()))
            A = random.choice(self.states_and_actions_taken[S])


            R, S_prime = self.Model[S][A]


            # if S == (0,8):
            #     print(S)

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + self.get_max(S_prime) - self.Q[S][A]
            )

    def do_one_dyna_Q_iteration(self):
        self.env.reset()


        S = self.env.current_state
        R = 0
        S_prime = self.env.current_state
        terminated = 0
        truncated = 0


        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(0.2, S)

            self.update_state_action_collection(S, A)

            S_prime, R, terminated, truncated  = self.env.step(A)




            # if S == (0,8):
            #     print(S)
            self.Q[S][A] = self.Q[S][A] + 0.1* (R + 0.99999*self.get_max(S_prime) - self.Q[S][A])




            self.Model[S][A] = (R, S_prime)


            self.do_planning_updates()

        self.trajectories.append(trajectory)

    def do_dyna_Q(self):
        for i in tqdm(range(self.number_of_episodes)):
            self.do_one_dyna_Q_iteration()

In [363]:
maze = Maze()


In [364]:
dyna = Dyna_Q(maze, n=5, number_of_episodes=10000)

In [365]:
dyna.do_dyna_Q()

  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [00:15<00:00, 647.11it/s]


In [366]:
dyna.states_and_actions_taken[(1,6)]

['down', 'up', 'left', 'right']

In [367]:
dyna.Q[(1,6)]

{'up': 0.9999689195596971,
 'down': 0.999970451956215,
 'left': 0.9999683894135174,
 'right': 0.9999688242410503}

{(0, 0): {'up': 0.9999530729536148,
  'down': 0.9999514799632357,
  'left': 0.9999532620190756,
  'right': 0.9999534470658714},
 (0, 1): {'up': 0.9999554307775518,
  'down': 0.9999532165081219,
  'left': 0.9999535464873427,
  'right': 0.9999557026517105},
 (0, 2): {'up': 0.999958629789127,
  'down': 0.9999583265160135,
  'left': 0.9999558457343815,
  'right': 0.9999593453475721},
 (0, 3): {'up': 0.999962308250867,
  'down': 0.9999633816074919,
  'left': 0.9999595730505751,
  'right': 0.9999627540974351},
 (0, 4): {'up': 0.9999659781005864,
  'down': 0.9999655337167402,
  'left': 0.9999633590707664,
  'right': 0.9999660122098636},
 (0, 5): {'up': 0.9999673439445745,
  'down': 0.9999675751383114,
  'left': 0.9999661373326147,
  'right': 0.9999677190368479},
 (0, 6): {'up': 0.9999688829026371,
  'down': 0.9999684065884658,
  'left': 0.9999676024623426,
  'right': 0.9999688817295614},
 (0, 7): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 8): {'up': 0, 'down': 0, 'left': 0, 'right': 0}

In [359]:
0.1 * (0 + 0.9999*0.9715 - 0.969)

0.00024028500000000674

In [355]:
dyna.Q[(2,6)]

{'up': 0.9711044503767032,
 'down': 0.9715994432250531,
 'left': 0.9708552745575395,
 'right': 0.9716521502291579}

In [353]:
dyna.Q[(1,7)]

{'up': 0, 'down': 0, 'left': 0, 'right': 0}

In [370]:
class PlotMaze():
    def __init__(self, maze, policy):
        self.maze = maze
        self.policy = policy

In [369]:
dyna.get_policy()

{(0, 0): 'right',
 (0, 1): 'right',
 (0, 2): 'right',
 (0, 3): 'down',
 (0, 4): 'right',
 (0, 5): 'right',
 (0, 6): 'up',
 (0, 7): 'Terminal',
 (0, 8): 'Terminal',
 (1, 0): 'right',
 (1, 1): 'up',
 (1, 2): 'Terminal',
 (1, 3): 'down',
 (1, 4): 'right',
 (1, 5): 'right',
 (1, 6): 'down',
 (1, 7): 'Terminal',
 (1, 8): 'up',
 (2, 0): 'right',
 (2, 1): 'up',
 (2, 2): 'Terminal',
 (2, 3): 'right',
 (2, 4): 'down',
 (2, 5): 'down',
 (2, 6): 'down',
 (2, 7): 'Terminal',
 (2, 8): 'up',
 (3, 0): 'right',
 (3, 1): 'down',
 (3, 2): 'Terminal',
 (3, 3): 'right',
 (3, 4): 'right',
 (3, 5): 'right',
 (3, 6): 'right',
 (3, 7): 'right',
 (3, 8): 'right',
 (4, 0): 'down',
 (4, 1): 'right',
 (4, 2): 'right',
 (4, 3): 'up',
 (4, 4): 'right',
 (4, 5): 'Terminal',
 (4, 6): 'right',
 (4, 7): 'right',
 (4, 8): 'up',
 (5, 0): 'right',
 (5, 1): 'right',
 (5, 2): 'right',
 (5, 3): 'right',
 (5, 4): 'down',
 (5, 5): 'right',
 (5, 6): 'up',
 (5, 7): 'up',
 (5, 8): 'up'}

In [244]:
dyna.Q[(0,8)]

{'up': 0, 'down': 0, 'left': 0, 'right': 0}

In [280]:
dyna.Q[(1,8)]

{'up': 0.9999999999999996,
 'down': 0.9932153499041008,
 'left': 0.9997256316545597,
 'right': 0.999088799533023}

In [298]:
dyna.Q[(0,8)]

{'up': 0, 'down': 0, 'left': 0, 'right': 0}

In [278]:
dyna.trajectories[-1]

[(2, 0),
 (2, 1),
 (1, 1),
 (1, 0),
 (1, 0),
 (0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (1, 4),
 (1, 5),
 (2, 5),
 (3, 5),
 (3, 6),
 (4, 6),
 (4, 7),
 (4, 8),
 (3, 8),
 (3, 7),
 (4, 7),
 (5, 7),
 (4, 7),
 (4, 8),
 (5, 8),
 (4, 8),
 (3, 8),
 (3, 8),
 (2, 8),
 (1, 8)]

In [41]:
random.choice(dyna.get_argmax((0,1)))

'left'

In [56]:
dyna.do_dyna_Q()

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 571.43it/s]


In [57]:
dyna.Q

{(0, 0): {'up': 0.1630186137991374,
  'down': 0.2835836711135685,
  'left': 0.15044080493190395,
  'right': 0.20989942578003715},
 (0, 1): {'up': 0.2998753221627383,
  'down': 0.4560407004362261,
  'left': 0.14628579813187037,
  'right': 0.16871195248357165},
 (0, 2): {'up': 0.11948885684781277,
  'down': 0.10758407495194981,
  'left': 0.20505097953508394,
  'right': 0.27587852455863426},
 (0, 3): {'up': 0.2700903078745973,
  'down': 0.4519709034998469,
  'left': 0.1472574384477691,
  'right': 0.3115428070756694},
 (0, 4): {'up': 0.49208075125324713,
  'down': 0.6950700333257869,
  'left': 0.24400458885186135,
  'right': 0.48270310379473363},
 (0, 5): {'up': 0.5024960092334178,
  'down': 0.6773907764182368,
  'left': 0.4378772917422895,
  'right': 0.4801535243781293},
 (0, 6): {'up': 0.4951493142208294,
  'down': 0.6995333021851355,
  'left': 0.4611605527375689,
  'right': 0.45977495385850325},
 (0, 7): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 8): {'up': 0.21618332658468,
  'd

In [13]:
dyna.Q[(0,0)]["down"] = 1

In [532]:
dyna.Q[(1,0)]["right"] = 4

In [541]:
dyna.update_state_action_collection((2,0),"up")

In [542]:
dyna.states_and_actions_taken

{(3, 0): ['left', 'up', 'down'],
 (3, 1): ['down'],
 (1, 0): ['down'],
 (2, 0): ['down', 'up']}

In [554]:
dyna.Model[(2, 0)]["up"] = (0, (1, 0))

In [555]:
dyna.Model

{(0, 0): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 1): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 2): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 3): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 4): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 5): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 6): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 7): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 8): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 0): {'up': (0, 0), 'down': (0, (2, 0)), 'left': (0, 0), 'right': (0, 0)},
 (1, 1): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 2): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 3): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 4): {'up': (0, 0), 'down': (

In [556]:
dyna.do_planning_updates()

In [557]:
dyna.Q

{(0, 0): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 1): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 2): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 3): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 4): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 5): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 6): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 7): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 8): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 0): {'up': 0, 'down': 0, 'left': 0, 'right': 4},
 (1, 1): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 2): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 3): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 4): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 5): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 6): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 7): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 8): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (2, 0): {'up': 0, 'down': 0

In [326]:
dyna.get_argmax((1,0))

'right'

In [481]:
dyna.get_action(0.2, (1,0))

'right'

In [280]:
dyna.Q[(1,0)]

{'up': 0, 'down': 0, 'left': 0, 'right': 4}